# JMFFT-8.0 Library

Install the libjmfft.a library

In [9]:
%%writefile JMFFT-8.0/arch/Make.inc.gnu
FC = gfortran
FFLAGS = -O3 -fdefault-real-8 -fdefault-double-8
LD = f90
LDFLAGS =
AR  = ar
ARFLAGS = rv
RM      = rm
RMFLAGS = -f

Writing JMFFT-8.0/arch/Make.inc.gnu


In [10]:
%%bash
cd JMFFT-8.0/arch
rm Make.inc
ln -s Make.inc.gnu Make.inc
cd ..

Changes
* Comment "SHELL = /bin/ksh" in Makefile
* Put "copyright = .true." in jmccm1d.f90

### Libray build

In [ ]:
%%bash
cd JMFFT-8.0
make

### Adapted from tjmscfft.f90

In [131]:
%%writefile test01.f90
! Real-complex Fourier transform 1D

program tjmscfft
    implicit none
    
    ! n - INTEGER type scalar. (input)
    integer, parameter :: n = 6
    ! SCFFT calculates the FFT of the real vector x (input)
    real, dimension(0:n-1) :: x, xx
    ! returns the result in the complex vector y
    complex, dimension(0:n/2) :: y
    ! to store cosines and sines
    integer, parameter :: ntable = 100+2*n
    real, dimension(0:ntable-1) :: table
    ! actually jm routines only need 2 * n
    integer, parameter :: nwork = 4+4*n
    real, dimension(nwork) :: work
    ! isign - initialize coefficient table, or apply FFT
    integer :: isign
    ! scale - each element of the vector y is multiplied
    real :: scale
    integer :: isys    ! not used
    integer :: i, j
    real :: twopi
    complex :: s

    ! prepare the input table
    call random_number( x )
    write(*,'(10e10.2)') x
    xx = x
    print*

    scale = 1.
    isys = 0

    ! 0 : initializes the array table and returns its value,
    !     only isign, n and table are checked and used
    isign = 0
    call scfft(isign, n, scale, x, y, table, work, isys)
    write(*,'(10e10.2)') table
    print*
    
    ! 1 : apply an FFT
    isign = 1
    print*, 'jmscfft ', n, isign, scale
    print*
    call scfft(isign,n,scale,x,y,table,work,isys)

    ! print the output table
    write(*, "(*('('spe0.2xspe0.2'j)':x))") y
    print*

    ! what to find
    twopi = 2 * acos(real(-1))
    ! repair the input table
    x = xx
    ! and calculate
    do i = 0, n/2
        s = 0
        do j = 0, n-1
            s = s + cmplx(cos(twopi*i*j/real(n)), &
                          isign*sin(twopi*i*j/real(n)))*x(j)
        end do
        write(*,"(*('('spe0.2xspe0.2'j) ':x))", advance='no') s*scale
    end do

end program tjmscfft

Overwriting test01.f90


In [132]:
! gfortran -O3 -o test01 test01.f90 -L JMFFT-8.0/lib -l jmfft

In [133]:
! ./test01

  0.97E-01  0.80E+00  0.42E+00  0.45E+00  0.46E+00  0.86E+00

  0.00E+00  0.24E+01  0.00E+00  0.20E+01  0.00E+00  0.19E+01  0.00E+00  0.21E+01  0.00E+00  0.19E+01
  0.00E+00  0.19E+01  0.00E+00  0.00E+00  0.70E-44  0.00E+00  0.77E+27  0.46E-40  0.78E+27  0.46E-40
  0.14E-44  0.00E+00  0.10E-37  0.00E+00  0.84E-44  0.00E+00  0.11E-40  0.92E-40  0.45E-43  0.00E+00
  0.17E-37  0.00E+00  0.51E+27  0.46E-40  0.00E+00  0.00E+00  0.00E+00 -0.17E+39  0.00E+00  0.00E+00
  0.36E-42  0.00E+00  0.11E-40  0.92E-40  0.16E-09  0.16E-09  0.16E-09  0.16E-09  0.00E+00  0.00E+00
  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00
  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.23E-37  0.00E+00  0.91E-40  0.00E+00 -0.17E+39
  0.00E+00  0.00E+00  0.36E-42  0.00E+00  0.00E+00  0.00E+00  0.16E-09  0.16E-09  0.16E-09  0.16E-09
  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.00E+00  0.58E+27  0.46E-40
  0.65E-26  0.00E+00  0.58E+2